<a href="https://colab.research.google.com/github/Bishre313/testing/blob/main/Data_preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Libraries

In [ ]:
import pandas as pd
import numpy as np
import io
from google.colab import drive

Loading Data

In [ ]:
drive.mount('/content/drive')
uploaded_file='/content/drive/MyDrive/DSA ICT/Data/ved.csv'
car=pd.read_csv(uploaded_file)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#EDA

In [ ]:
car.head()

,DayNum,VehId,Trip,Timestamp(ms),Latitude[deg],Longitude[deg],Vehicle Speed[km/h],MAF[g/sec],Engine RPM[RPM],Absolute Load[%],...,Energy_Consumption,Matchted Latitude[deg],Matched Longitude[deg],Match Type,Class of Speed Limit,Speed Limit[km/h],Speed Limit with Direction[km/h],Intersection,Bus Stops,Focus Points
0,57.05024,139,1163,0,42.302634,-83.704336,16.0,20.770000,1934.0,56.470589,...,0.001968,42.302591,-83.704333,0,0.0,72,72.0,NaN,NaN,NaN
1,57.05024,139,1163,900,42.302634,-83.704336,23.0,18.240000,1776.0,56.470589,...,0.002485,42.302591,-83.704333,1,0.0,72,72.0,NaN,NaN,NaN
2,57.05024,139,1163,1900,42.302634,-83.704336,29.0,18.240000,1776.0,56.470589,...,0.003133,42.302591,-83.704333,1,0.0,72,72.0,NaN,NaN,NaN
3,57.05024,139,1163,2900,42.302634,-83.704336,32.0,20.809999,1772.0,56.470589,...,0.003944,42.302591,-83.704333,1,0.0,72,72.0,NaN,NaN,NaN
4,57.05024,139,1163,3000,42.302610,-83.704049,32.0,20.809999,1772.0,56.470589,...,0.003944,42.302600,-83.704049,0,0.0,72,72.0,NaN,NaN,NaN


In [ ]:
car.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 35 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   DayNum                            20000 non-null  float64
 1   VehId                             20000 non-null  int64  
 2   Trip                              20000 non-null  int64  
 3   Timestamp(ms)                     20000 non-null  int64  
 4   Latitude[deg]                     20000 non-null  float64
 5   Longitude[deg]                    20000 non-null  float64
 6   Vehicle Speed[km/h]               20000 non-null  float64
 7   MAF[g/sec]                        13396 non-null  float64
 8   Engine RPM[RPM]                   13602 non-null  float64
 9   Absolute Load[%]                  10123 non-null  float64
 10  OAT[DegC]                         8447 non-null   float64
 11  Fuel Rate[L/hr]                   0 non-null      float64
 12  Air 

#Pipeline

In [ ]:

def run_preprocessing(uploaded_file):
   #AETS preprocessing pipeline.


    print("--- STEP 1: LOADING RAW DATA ---")

    df = pd.read_csv(uploaded_file)
    print(f"Data Loaded successfully. Shape: {df.shape}")
    print("Sample Rows (Original):")
    print(df[['Engine RPM[RPM]', 'Vehicle Speed[km/h]']].head())
    print("-" * 50)




    print("\n--- STEP 2: COLUMN SELECTION & FILTERING ---")
    # We only care about signals relevant to Engine Performance & Physics
    essential_cols = [
        'Vehicle Speed[km/h]',
        'Engine RPM[RPM]',
        'OAT[DegC]',
        'Gradient',
        'Fuel Rate[L/hr]'
    ]

    # Check if any columns are missing (robustness check)
    existing_cols = [c for c in essential_cols if c in df.columns]
    df = df[existing_cols].copy()
    print(f"Columns selected. New Shape: {df.shape}")
    print("-" * 50)




    print("\n---  STEP 3: HANDLING NULLS & SENSOR ERRORS ---")
    # We cannot optimize if we don't know the Speed or RPM

    initial_rows = len(df)
    df = df.dropna(subset=['Engine RPM[RPM]', 'Vehicle Speed[km/h]'])
    dropped = initial_rows - len(df)
    print(f"Dropped {dropped} rows with missing essential sensor data.")

    # Fill missing Environmental data (like Gradient and Temperature)
    df['Gradient'] = df['Gradient'].fillna(0) # Assume flat road if unknown
    df['OAT[DegC]'] = df['OAT[DegC]'].fillna(df['OAT[DegC]'].mean()) # Use average temp if missing
    print("Environmental data cleaned (Gradient and OAT).")
    print("-" * 50)




    print("\n--- STEP 4: FEATURE ENGINEERING (ACCELERATION) ---")
    # Acceleration is critical because it determines 'Inertial Demand' on the engine
    # Formula: a = dv/dt
    # We calculate the delta change in speed between consecutive time steps
    df['Acceleration'] = df['Vehicle Speed[km/h]'].diff().fillna(0)
    print("Acceleration calculated from Speed deltas.")
    print(df[['Vehicle Speed[km/h]', 'Acceleration']].head())
    print("-" * 50)





    print("\n--- STEP 5: SYNTHETIC FUEL CALCULATION (PHYSICS MODEL) ---")
    """
    REASONING: Many telematics datasets like VED have gaps in Fuel Rate reporting.
    To ensure the simulation continues, we calculate a 'Power Proxy' using physics:
    Power = (Inertia + Drag + Hill Climbing) * Velocity
    Fuel Consumption = (Friction Loss from RPM) + (Power Demand)
    """

    k_friction = 0.0005 # L/hr per RPM (calibrated for 2.0L engine)
    k_power = 0.05      # L/hr per Unit Power Proxy

    power_demand = (
        df['Vehicle Speed[km/h]'] * df['Acceleration'].clip(lower=0) +   # Inertia
        0.01 * df['Vehicle Speed[km/h]']**2 +                            # Aerodynamic Drag
        9.8 * df['Vehicle Speed[km/h]'] * np.sin(np.radians(df['Gradient'])) # Gravity/Hill Climbing
    )

    df['Synthetic_Fuel'] = (k_friction * df['Engine RPM[RPM]'] + k_power * power_demand.clip(lower=0))

    # Use Synthetic Fuel if Actual Fuel is missing or zero
    if 'Fuel Rate[L/hr]' not in df.columns or df['Fuel Rate[L/hr]'].sum() == 0:
        print("Warning: Actual Fuel data missing. Applying Physics-based Synthetic Fuel Model.")
        df['Fuel Rate[L/hr]'] = df['Synthetic_Fuel']
    else:
        print("Actual Fuel Rate detected. Merging with Synthetic data for validation.")

    print("-" * 50)




    print("\n---  STEP 6: FINAL DATA VALIDATION ---")
    print(f"Preprocessing Complete. Final Record Count: {len(df)}")
    print("\nDescriptive Statistics for Preprocessed Features:")
    print(df.describe().loc[['mean', 'min', 'max']])
    return df


In [ ]:
run_preprocessing(uploaded_file)

--- STEP 1: LOADING RAW DATA ---
Data Loaded successfully. Shape: (20000, 35)
Sample Rows (Original):
   Engine RPM[RPM]  Vehicle Speed[km/h]
0           1934.0                 16.0
1           1776.0                 23.0
2           1776.0                 29.0
3           1772.0                 32.0
4           1772.0                 32.0
--------------------------------------------------

--- STEP 2: COLUMN SELECTION & FILTERING ---
Columns selected. New Shape: (20000, 5)
--------------------------------------------------

---  STEP 3: HANDLING NULLS & SENSOR ERRORS ---
Dropped 6398 rows with missing essential sensor data.
Environmental data cleaned (Gradient and OAT).
--------------------------------------------------

--- STEP 4: FEATURE ENGINEERING (ACCELERATION) ---
Acceleration calculated from Speed deltas.
   Vehicle Speed[km/h]  Acceleration
0                 16.0           0.0
1                 23.0           7.0
2                 29.0           6.0
3                 32.0    

,Vehicle Speed[km/h],Engine RPM[RPM],OAT[DegC],Gradient,Fuel Rate[L/hr],Acceleration,Synthetic_Fuel
0,16.0,1934.0,25.520986,0.00000,1.095000,0.0,1.095000
1,23.0,1776.0,25.520986,0.00000,9.202500,7.0,9.202500
2,29.0,1776.0,25.520986,0.00000,10.008500,6.0,10.008500
3,32.0,1772.0,25.520986,-0.00108,6.197704,3.0,6.197704
4,32.0,1772.0,25.520986,0.00000,1.398000,0.0,1.398000
...,...,...,...,...,...,...,...
19995,5.0,584.0,25.520986,0.00000,0.304500,-22.0,0.304500
19996,5.0,584.0,25.520986,0.00000,0.304500,0.0,0.304500
19997,2.0,602.0,25.520986,0.00000,0.303000,-3.0,0.303000
19998,2.0,594.0,25.520986,0.00000,0.299000,0.0,0.299000
